In [1]:
# 1. 라이브러리 설치
!pip install -q gradio

import gradio as io

# 2. 핵심 로직 함수 (유동적 지출을 고려한 시나리오 연산)
def calculate_flexible_balance(current_balance, target_balance, hourly_wage, weekly_hours, normal_weekly_exp, max_weekly_exp, monthly_expense):
    try:
        # 입력값 전처리 (콤마 및 공백 제거)
        current_bal = float(str(current_balance).replace(",", "").strip())
        target_bal = float(str(target_balance).replace(",", "").strip())
        wage = float(str(hourly_wage).replace(",", "").strip())
        hours = float(str(weekly_hours).replace(",", "").strip())
        normal_exp = float(str(normal_weekly_exp).replace(",", "").strip())
        max_exp = float(str(max_weekly_exp).replace(",", "").strip())
        month_exp = float(str(monthly_expense).replace(",", "").strip())

        if max_exp < normal_exp:
            return "❌ '과소비하는 주의 지출'은 '평소 지출'보다 커야 합니다!"

        # [연산 1] 월 수입 계산
        monthly_income = wage * hours * 4.345
        shortage = target_bal - current_bal

        # [연산 2] 시나리오 A: 평소처럼 살 때 (최선)
        total_exp_normal = (normal_exp * 4.345) + month_exp
        net_normal = monthly_income - total_exp_normal

        # [연산 3] 시나리오 B: 매주 과소비할 때 (최악)
        total_exp_max = (max_exp * 4.345) + month_exp
        net_max = monthly_income - total_exp_max

        # 결과 메시지 조립 [출력]
        result = "📊 [현재 재정 상태 요약]\n"
        result += f"💰 예상 월 수입: {int(monthly_income):,}원\n"
        result += f"🎯 목표까지 부족한 돈: {int(shortage):,}원\n\n"

        result += "━━━━━━ 미래 잔고 시나리오 예측 ━━━━━━\n\n"

        # 시나리오 A (최선) 출력
        result += "🟢 1. 평소대로 아껴 쓸 때 (주 {:,}원 지출)\n".format(int(normal_exp))
        if net_normal <= 0:
            result += "   👉 평소 지출만으로도 이미 적자입니다. 목표 달성 불가능!\n"
        else:
            months_normal = shortage / net_normal
            result += f"   👉 매달 약 {int(net_normal):,}원씩 저축 가능 ➡️ 딱 ' {months_normal:.1f}개월 ' 뒤 목표 달성!\n"

        result += "\n"

        # 시나리오 B (최악) 출력
        result += "🔴 2. 고삐 풀려 과소비할 때 (주 {:,}원 지출)\n".format(int(max_exp))
        if net_max <= 0:
            result += f"   👉 매달 약 {int(abs(net_max)):,}원씩 [적자]가 납니다. 통장 잔고가 늘어나긴커녕 줄어듭니다!\n"
        else:
            months_max = shortage / net_max
            result += f"   👉 저축액이 {int(net_max):,}원으로 토막 남 ➡️ 목표 달성까지 ' {months_max:.1f}개월 '로 대폭 늘어남!\n"

        result += "\n⚠️ [한줄평]\n"
        if net_max <= 0 and net_normal > 0:
            result += "정신 차리셔야 합니다. 주간 지출이 유동적이라는 핑계로 과소비하는 주가 많아지면, 당신의 목표 달성일은 '영원히 오지 않는 날'이 됩니다."
        elif net_max > 0 and net_normal > 0:
            result += "돈을 많이 쓰는 주가 생기더라도 흑자는 유지되네요! 다만 소비를 줄일수록 목표 달성이 훨씬 빨라집니다."
        else:
            result += "비상사태입니다. 알바 시간을 늘리거나 당장 고정 지출을 구조조정 하세요."

        return result

    except ValueError:
        return "❌ 모든 입력창에는 숫자만 정확하게 입력해 주세요!"

# 3. 화면 UI 디자인 및 실행 (Gradio 인터페이스)
with io.Blocks(theme=io.themes.Soft()) as demo:
    io.Markdown("# 📉 유동적 지출 반영 잔고 수호기")
    io.Markdown("매주 나가는 돈이 달라 걱정이신가요? 평소 지출과 과소비할 때의 지출을 모두 넣어 미래를 예측해 보세요.")

    with io.Row():
        with io.Column():
            io.Markdown("### 🏦 계좌 및 목표")
            current_input = io.Textbox(label="현재 내 계좌 잔고 (원)", placeholder="예: 300000")
            target_input = io.Textbox(label="최종 목표 계좌 잔고 (원)", placeholder="예: 2000000")

            io.Markdown("### 💸 알바 수입 (고정)")
            wage_input = io.Textbox(label="내 알바 시급 (원)", value="10030")
            hours_input = io.Textbox(label="일주일에 일하는 총 시간 (시간)", placeholder="예: 15")

            io.Markdown("### 🛒 유동적인 주간 소비 입력")
            normal_exp_input = io.Textbox(label="평소처럼 쓰는 주의 지출 (원)", placeholder="예: 50000")
            max_exp_input = io.Textbox(label="약속이나 쇼핑 등으로 과소비하는 주의 지출 (원)", placeholder="예: 120000")

            io.Markdown("### 🏠 월간 고정 지출")
            month_exp_input = io.Textbox(label="한 달에 무조건 나가는 돈 (원) (숨만 쉬어도 나가는 월세, 통신비 등)", value="0")

            btn = io.Button("🔥 시나리오별 미래 예측하기", variant="primary")

        with io.Column():
            io.Markdown("### 📝 최선 vs 최악 시나리오 분석 결과")
            output_box = io.Textbox(label="연산 결과창", lines=18)

    # 버튼 클릭 시 함수 실행
    btn.click(
        fn=calculate_flexible_balance,
        inputs=[current_input, target_input, wage_input, hours_input, normal_exp_input, max_exp_input, month_exp_input],
        outputs=output_box
    )

# 웹앱 실행 및 외부 공유 링크 생성
demo.launch(share=True)

/tmp/ipykernel_1369/2756954533.py:72: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with io.Blocks(theme=io.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://169b7446e82967124a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
